## Import Required Libraries and Project Folder Paths

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

from FDApy import IrregularFunctionalData
from FDApy.representation import (
    DenseArgvals,
    IrregularArgvals,
    IrregularValues
)
from FDApy.preprocessing import UFPCA


# Project paths
ROOT_DIR = Path("../..").resolve()

DATA_DIR = ROOT_DIR / "data"

RESULTS_DIR = (
    ROOT_DIR
    / "results"
    / "module_2"
    / "dataset_creation"
)

PLOTS_DIR = (
    ROOT_DIR
    / "plots"
    / "module_2"
    / "dataset_creation"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


print("Root directory:", ROOT_DIR)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Plots directory:", PLOTS_DIR)

Root directory: C:\Users\samsa\Documents\ICU Clustering
Data directory: C:\Users\samsa\Documents\ICU Clustering\data
Results directory: C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation
Plots directory: C:\Users\samsa\Documents\ICU Clustering\plots\module_2\dataset_creation


#### Observation :
- Imports the required libraries for data manipulation, scaling, file handling and functional data analysis.
- The use of FDApy shows that the notebook is prepared to analyze irregularly recorded patient time-series data which is common in healthcare and ICU datasets.
- Sets up separate directories for data, results and plots helping keep the workflow organized.
- Automatically creates missing folders, supporting a more reproducible and error-resistant for the later analysis.

## Load the Patient Table

We only load the patient columns needed for cohort construction and demographic features

In [2]:
patient_columns = [
    "patientunitstayid",
    "uniquepid",
    "unitvisitnumber",
    "hospitaladmitoffset",
    "apacheadmissiondx",
    "age",
    "gender",
    "ethnicity",
    "unittype",
    "admissionheight",
    "admissionweight"
]

patient = pd.read_csv(
    DATA_DIR / "patient.csv",
    usecols=patient_columns
)

print("Patient table shape:", patient.shape)

print("\nColumns loaded:")
print(patient.columns.tolist())

print(
    "\nUnique ICU stays:",
    patient["patientunitstayid"].nunique()
)

print(
    "Unique patients:",
    patient["uniquepid"].nunique()
)

Patient table shape: (200859, 11)

Columns loaded:
['patientunitstayid', 'gender', 'age', 'ethnicity', 'apacheadmissiondx', 'admissionheight', 'hospitaladmitoffset', 'unittype', 'unitvisitnumber', 'admissionweight', 'uniquepid']

Unique ICU stays: 200859
Unique patients: 139367


#### Observation :
- Loads the patient-level eICU dataset while selecting only $11$ clinically relevent variables, improving efficiency by avoiding unnecessary columns.
- The dataset contains $200,859$ ICU stays corresponding to $139,367$ unique patients confirming that some patients experienced multiple ICU admissions/stays
- The selected variables capture important information on patient identity, demographics, ICU characteristics, admission details and APACHE admission diagnosis.
- This block establishes the initial patient population from which the sepsis cohort will subsequently be identified and filtered.

## Identify the Sepsis Cohort

In [3]:
sepsis = patient[
    patient["apacheadmissiondx"]
    .astype(str)
    .str.contains(
        "sepsis",
        case=False,
        na=False
    )
].copy()

print("Initial sepsis ICU admissions:", len(sepsis))

print(
    "Unique sepsis patients:",
    sepsis["uniquepid"].nunique()
)

print("\nSepsis admission diagnosis categories:")
print(
    sepsis["apacheadmissiondx"]
    .value_counts()
)

Initial sepsis ICU admissions: 23136
Unique sepsis patients: 20131

Sepsis admission diagnosis categories:
apacheadmissiondx
Sepsis, pulmonary                        8862
Sepsis, renal/UTI (including bladder)    5273
Sepsis, GI                               2881
Sepsis, unknown                          2602
Sepsis, cutaneous/soft tissue            1933
Sepsis, other                            1510
Sepsis, gynecologic                        75
Name: count, dtype: int64


#### Observation :
- Identifies the sepsis cohort by filtering `apacheadmissiondx` for diagnoses containing the term `sepsis` regardless of capitalization.
- This results in $23,136$ sepsis ICU admissions involving $20,131$ unique patients showing that some patients had more than one sepsis-related ICU stay.
- The sepsis cases are further summarized by infection source, with pulmonary sepsis being the most common followed by renal/UTI and gastrointestinal sepsis.
- This block establishes the initial sepsis-specific population for the subsequent cohort-selection steps.

## Retain One Sepsis ICU Stay Per Patient

In [4]:
cohort = sepsis.copy()

cohort = cohort.sort_values(
    [
        "uniquepid",
        "unitvisitnumber",
        "hospitaladmitoffset"
    ],
    ascending=[
        True,
        True,
        False
    ]
)

cohort = cohort.drop_duplicates(
    subset="uniquepid",
    keep="first"
).copy()

print(
    "Sepsis ICU stays before:",
    len(sepsis)
)

print(
    "ICU stays after retaining one per patient:",
    len(cohort)
)

print(
    "Unique patients:",
    cohort["uniquepid"].nunique()
)

print(
    "Duplicate patients remaining:",
    cohort["uniquepid"].duplicated().sum()
)

Sepsis ICU stays before: 23136
ICU stays after retaining one per patient: 20131
Unique patients: 20131
Duplicate patients remaining: 0


#### Observation :
- Ensures that each patient contributes only one sepsis ICU stay, preventing repeated admissions from the same individual from biasing the analysis.
- Patients are ordered using patient ID, ICU visit number and hospital admission offset before duplicates are removed.
- The cohort decreases from $23,136$ sepsis ICU stays to $20,131$ patients exactly matching the number of unique sepsis patients.
- The final check confirms zero duplicate patients creating a patient-level cohort suitable for downstream analysis.

## Apply `age`, `ICU type` and demographic criteria

In [5]:
cohort_filtered = cohort.copy()

# Convert age to numeric
cohort_filtered["age_numeric"] = (
    cohort_filtered["age"]
    .replace("> 89", "90")
    .astype(float)
)

# Adults only
cohort_filtered = cohort_filtered[
    cohort_filtered["age_numeric"] >= 18
].copy()

# Exclude cardiothoracic surgical ICU
cohort_filtered = cohort_filtered[
    cohort_filtered["unittype"] != "CSICU"
].copy()

# Require core demographic information
required_demographics = [
    "age",
    "gender",
    "ethnicity",
    "unittype"
]

cohort_filtered = cohort_filtered[
    ~cohort_filtered[
        required_demographics
    ].isna().any(axis=1)
].copy()

print(
    "Patients remaining after demographic criteria:",
    len(cohort_filtered)
)

print(
    "Unique patients:",
    cohort_filtered["uniquepid"].nunique()
)

print("\nAge summary:")
print(
    cohort_filtered["age_numeric"].describe()
)

print("\nICU type distribution:")
print(
    cohort_filtered["unittype"].value_counts()
)

Patients remaining after demographic criteria: 19605
Unique patients: 19605

Age summary:
count    19605.000000
mean        66.023310
std         16.342954
min         18.000000
25%         56.000000
50%         68.000000
75%         79.000000
max         90.000000
Name: age_numeric, dtype: float64

ICU type distribution:
unittype
Med-Surg ICU    13342
MICU             2621
Cardiac ICU      1189
CCU-CTICU        1166
SICU              754
Neuro ICU         386
CTICU             147
Name: count, dtype: int64


#### Observation :
- Applies the main demographic and ICU eligibility criteria to refine the sepsis cohort.
- Converts age into a numeric format, treating patients recorded as $\gt 89$ as age $90$ and restricts the cohort to adults aged $18$ years or older.
- Excludes patients admitted to the CSICU, helping maintain a more clinically comparable ICU population.
- Requires complete information for key variables : `age`, `gender`, `ethnicity` and `ICU type`
- After these filters, $19,605$ unique patients remain with a median age of $68$ years.
- Most patients were admitted to a Med-Surg ICU making it the dominant ICU setting in the filtered cohort.

## Load the Selected First-24-Hour Laboratory Measurements

In [6]:
selected_labs = [
    "bicarbonate",
    "lactate",
    "potassium",
    "platelets x 1000",
    "anion gap",
    "chloride",
    "BUN",
    "creatinine",
    "sodium",
    "glucose",
    "WBC x 1000",
    "Hgb"
]

lab = pd.read_csv(
    DATA_DIR / "lab.csv",
    usecols=[
        "patientunitstayid",
        "labresultoffset",
        "labname",
        "labresult"
    ]
)

candidate_ids = set(
    cohort_filtered["patientunitstayid"]
)

lab_24h = lab[
    (lab["patientunitstayid"].isin(candidate_ids))
    & (lab["labresultoffset"] >= 0)
    & (lab["labresultoffset"] <= 1440)
    & (lab["labname"].isin(selected_labs))
].copy()

valid_lab_24h = lab_24h[
    lab_24h["labresult"].notna()
].copy()

print(
    "Valid selected lab measurements in first 24 hours:",
    len(valid_lab_24h)
)

print(
    "Patients with at least one selected lab:",
    valid_lab_24h["patientunitstayid"].nunique()
)

print("\nMeasurements by laboratory:")
print(
    valid_lab_24h["labname"]
    .value_counts()
    .reindex(selected_labs)
)

Valid selected lab measurements in first 24 hours: 339694
Patients with at least one selected lab: 18737

Measurements by laboratory:
labname
bicarbonate         28161
lactate             22766
potassium           34877
platelets x 1000    24044
anion gap           23986
chloride            30254
BUN                 29973
creatinine          30110
sodium              33110
glucose             30783
WBC x 1000          23891
Hgb                 27739
Name: count, dtype: int64


#### Observation :
- Selects $12$ clinically important laboratory variables including lactate, creatinine, BUN, WBC, haemoglobin, platelets, electrolytes and glucose.
- Restricts laboratory data to the first $24$ hours of ICU admission $(0-1440 \text{ minutes})$ ensuring measurements reflect the patient's early clincial condition.
- Keeps only patients from the previously filtered sepsis cohort and removes laboratory records with missing results.
- A total of $339,694$ valid laboratory measurements are retained from $18,737$ patients.
- Laboratory measurement frequency varies across tests. For example, potassium and sodium are measured more frequently than lactate reflecting differences in routine clinical monitoring.
- This block prepares the early laboratory dataset for the subsequent completeness and longitudinal analysis.

## Keep patients with all $12$ selected laboratory variables

In [7]:
lab_availability = (
    valid_lab_24h
    .groupby(
        [
            "patientunitstayid",
            "labname"
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=selected_labs,
        fill_value=0
    )
)

has_all_labs = (
    lab_availability > 0
).all(axis=1)

patients_with_all_labs = set(
    lab_availability.index[
        has_all_labs
    ]
)

print(
    "Patients before laboratory completeness check:",
    len(cohort_filtered)
)

print(
    "Patients with all 12 selected labs:",
    len(patients_with_all_labs)
)

print(
    "Patients missing at least one selected lab:",
    len(cohort_filtered) - len(patients_with_all_labs)
)

Patients before laboratory completeness check: 19605
Patients with all 12 selected labs: 8328
Patients missing at least one selected lab: 11277


#### Observation :
- Evaluates laboratory completeness at the patient level by checking whether each patient has at least one measurement for all $12$ selected laboratory variables within the first $24$ hours.
- Out of $19,605$ eligible patients, only $8,328$ patients have complete coverage across all $12$ laboratory tests.
- $11,277$ patients are missing atleast one required laboratory variable, indicating substantial incompleteness in the clinical data.
- This block therefore identifies the laboratory-complete patient subset that can be used for more consistent downstream modelling and analysis.

## Load first-24-hour vital signs

In [8]:
vital = pd.read_csv(
    DATA_DIR / "vitalPeriodic.csv",
    usecols=[
        "patientunitstayid",
        "observationoffset",
        "sao2",
        "heartrate",
        "respiration"
    ]
)

vital_24h = vital[
    (vital["patientunitstayid"].isin(candidate_ids))
    & (vital["observationoffset"] >= 0)
    & (vital["observationoffset"] <= 1440)
].copy()

print(
    "First-24-hour vital-sign rows:",
    len(vital_24h)
)

print(
    "Patients with vital-sign records:",
    vital_24h["patientunitstayid"].nunique()
)

print("\nPatients with at least one measurement:")

for column in [
    "sao2",
    "heartrate",
    "respiration"
]:
    count = (
        vital_24h.loc[
            vital_24h[column].notna(),
            "patientunitstayid"
        ]
        .nunique()
    )

    print(column, ":", count)

First-24-hour vital-sign rows: 4950613
Patients with vital-sign records: 19372

Patients with at least one measurement:
sao2 : 19197
heartrate : 19368
respiration : 18063


#### Observation :
- Extracts three key vital signs : $\text{oxygen saturation(SaO2)}, \text{heart rate and respiratory rate}$.
- Restricts the data to the first $24$ hours of ICU admission, keeping the analysis focused on the patient's early physiological condition.
- A total of $4,950,613$ vital-sign records are available from $19,372$ patients showing that vital signs are measured very frequently in the ICU.
- $\text{Heart rate}$ has the highest patient coverage ($19,368$ patients) while $\text{respiratory rate}$ has the lowest ($18,063$ patients).
- This block assess the availability and coverage of early vital-sign measurements before further patient selection and time-series analysis.

## Create the final cohort with complete lab and vital coverage

In [9]:
# Count available vital-sign measurements per patient
vital_counts = (
    vital_24h
    .groupby("patientunitstayid")[
        [
            "sao2",
            "heartrate",
            "respiration"
        ]
    ]
    .count()
)

# Require at least one measurement for all three vital signs
has_all_vitals = (
    vital_counts > 0
).all(axis=1)

patients_with_all_vitals = set(
    vital_counts.index[
        has_all_vitals
    ]
)

# Patients satisfying the vital-sign requirement
cohort_vital = cohort_filtered[
    cohort_filtered["patientunitstayid"]
    .isin(patients_with_all_vitals)
].copy()

# Also require all 12 selected laboratory variables
final_cohort = cohort_vital[
    cohort_vital["patientunitstayid"]
    .isin(patients_with_all_labs)
].copy()

print(
    "Patients before vital-sign criterion:",
    len(cohort_filtered)
)

print(
    "Patients with all 3 vital signs:",
    len(cohort_vital)
)

print(
    "Patients with all 3 vitals and all 12 labs:",
    len(final_cohort)
)

print(
    "Unique patients in final cohort:",
    final_cohort["uniquepid"].nunique()
)

Patients before vital-sign criterion: 19605
Patients with all 3 vital signs: 17919
Patients with all 3 vitals and all 12 labs: 7880
Unique patients in final cohort: 7880


#### Observation :
- Counts the available measurements of $\text{SaO2}, \text{heart rate and respiratory rate}$ for each patient during the first $24$ hours.
- Retains only patients who have atleast one valid measurement for all three vital signs reducing the cohort from $19,605$ to $17,919$ patients.
- Further intersects this group with patients having all $12$ required laboratory variables.
- The resulting final cohort contains $7,880$ unique patients, ensuring complete coverage of the selected laboratory and vital-sign variables for downstream analysis.

## Create demographic and admission features

In [10]:
demo_features = final_cohort[
    [
        "patientunitstayid",
        "age_numeric",
        "gender",
        "ethnicity",
        "admissionheight",
        "unittype",
        "admissionweight"
    ]
].copy()

demo_features = demo_features.rename(
    columns={
        "age_numeric": "age"
    }
)

print(
    "Demographic feature table shape:",
    demo_features.shape
)

print("\nMissing values:")
print(
    demo_features.isna().sum()
)

print("\nFirst 5 rows:")
print(
    demo_features.head()
)

Demographic feature table shape: (7880, 7)

Missing values:
patientunitstayid      0
age                    0
gender                 0
ethnicity              0
admissionheight       67
unittype               0
admissionweight      191
dtype: int64

First 5 rows:
       patientunitstayid   age  gender  ethnicity  admissionheight  \
1610              151900  66.0  Female  Caucasian            165.1   
10132             210208  52.0    Male      Asian            162.6   
5657              179269  82.0  Female  Caucasian            154.9   
4709              172764  71.0  Female  Caucasian            167.6   
3673              166175  60.0    Male  Caucasian            185.4   

           unittype  admissionweight  
1610           MICU             86.8  
10132          MICU             70.7  
5657   Med-Surg ICU             72.5  
4709   Med-Surg ICU             85.5  
3673   Med-Surg ICU            107.9  


#### Observation :
- Creates the demographic feature table for the final cohort of $7,880$ patients using age, gender, ethnicity, height, ICU type and admission weight.
- Renames `age_numeric` to age for clearer downstream use.
- Most demographic variables are complete but admission height has $67$ missing values and admission weight has $191$ missing values.
- This block prepares the patient-level baseline characteristics that can later be combined with laboratory and vital-sign features for analysis or modelling.

## Create laboratory summary features

For each of the $12$ selected labs, calculate the first-24-hour :
- minimum
- maximum
- mean
- standard deviation

In [11]:
final_ids = set(
    final_cohort["patientunitstayid"]
)

lab_final = valid_lab_24h[
    valid_lab_24h["patientunitstayid"]
    .isin(final_ids)
].copy()

lab_summary = (
    lab_final
    .groupby(
        [
            "patientunitstayid",
            "labname"
        ]
    )["labresult"]
    .agg([
        "min",
        "max",
        "mean",
        "std"
    ])
    .unstack()
)

lab_summary.columns = [
    f"{stat}_{lab_name}"
    for stat, lab_name in lab_summary.columns
]

lab_summary = lab_summary.reset_index()

print(
    "Laboratory summary table shape:",
    lab_summary.shape
)

print(
    "Number of laboratory features:",
    lab_summary.shape[1] - 1
)

print(
    "\nPatients represented:",
    lab_summary["patientunitstayid"].nunique()
)

print(
    "\nTotal missing values:",
    lab_summary.isna().sum().sum()
)

Laboratory summary table shape: (7880, 49)
Number of laboratory features: 48

Patients represented: 7880

Total missing values: 48729


#### Observation :
- Restricts the laboratory data to the final cohort of $7,880$ patients.
- Summarizes each of the $12$ laboratory variables using four statistics : minimum, maximum, mean and standard deviation during the first $24$ hours.
- Produces $48$ laboratory features in total, providing information on both the level and variability of each laboratory measurement.
- Although all $7,880$ patients are represented, the resulting table contains $48,729$ missing values, indicating that some summary statistics particularly standard deviations cannot be calculated for patients with limited repeated measurements.

## Check whether missing lab values come only from standard deviation

In [12]:
missing_by_column = (
    lab_summary
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_by_column = missing_by_column[
    missing_by_column > 0
]

print("Columns with missing values:")
print(missing_by_column)

print(
    "\nNumber of columns with missing values:",
    len(missing_by_column)
)

print(
    "\nDo all missing-value columns start with 'std_'?",
    all(
        col.startswith("std_")
        for col in missing_by_column.index
    )
)

Columns with missing values:
std_WBC x 1000          5096
std_platelets x 1000    5004
std_Hgb                 4590
std_anion gap           3980
std_BUN                 3942
std_bicarbonate         3933
std_chloride            3917
std_creatinine          3909
std_glucose             3883
std_lactate             3731
std_sodium              3624
std_potassium           3120
dtype: int64

Number of columns with missing values: 12

Do all missing-value columns start with 'std_'? True


#### Observation :
- Checks which of the $48$ laboratory features still contain missing values.
- Missing values appear only in the standard deviation features, not in the minimum, maximum or mean values.
- WBC, platelets and haemoglobin have the most missing standard deviations.
- This happens because some patient have only one measurement, so a standard deviation cannot be calculated.

## Verify that missing SD values correspond to single measurements

In [13]:
lab_observation_counts = (
    lab_final
    .groupby(
        [
            "patientunitstayid",
            "labname"
        ]
    )
    .size()
    .unstack()
)

comparison = []

for lab_name in selected_labs:

    one_measurement = (
        lab_observation_counts[lab_name] == 1
    ).sum()

    missing_std = (
        lab_summary[
            f"std_{lab_name}"
        ]
        .isna()
        .sum()
    )

    comparison.append(
        {
            "lab": lab_name,
            "one_measurement": one_measurement,
            "missing_std": missing_std,
            "match": one_measurement == missing_std
        }
    )

std_check = pd.DataFrame(comparison)

print(std_check)

print(
    "\nAll missing SD values explained by one measurement:",
    std_check["match"].all()
)

                 lab  one_measurement  missing_std  match
0        bicarbonate             3933         3933   True
1            lactate             3731         3731   True
2          potassium             3120         3120   True
3   platelets x 1000             5004         5004   True
4          anion gap             3980         3980   True
5           chloride             3917         3917   True
6                BUN             3942         3942   True
7         creatinine             3909         3909   True
8             sodium             3624         3624   True
9            glucose             3883         3883   True
10        WBC x 1000             5096         5096   True
11               Hgb             4590         4590   True

All missing SD values explained by one measurement: True


#### Observation :
- Checks whether the missing standard deviation values are caused by patients having only one measurement for a lab test.
- For all $12$ laboratory variables, the number of patients with one measurement exactly matches the number of missing standard deviations.
- Confirms that all missing standard deviation values are fully explained by single measurements.
- This supports handling those missing standard deviations consistently.

## Replace single-measurement laboratory SD values with 0

In [14]:
# Standard-deviation feature columns
std_columns = [
    col
    for col in lab_summary.columns
    if col.startswith("std_")
]

# Missing SD values were verified to come only from patients with one measurement
lab_summary[std_columns] = (
    lab_summary[std_columns]
    .fillna(0)
)

print(
    "Missing values remaining in lab summary:",
    lab_summary.isna().sum().sum()
)

print(
    "Laboratory summary shape:",
    lab_summary.shape
)

print(
    "Patients represented:",
    lab_summary["patientunitstayid"].nunique()
)

Missing values remaining in lab summary: 0
Laboratory summary shape: (7880, 49)
Patients represented: 7880


#### Observation :
- Replaces the missing laboratory standard deviation values with `0`.
- This is appropriate because the previous check confirmed these missing values occur only when a patient has one measurement, meaning no within-patient variability can be calculated.
- After this step, the laboratory summary contains no missing values.
- The final laboratory feature table contains $7,880$ patients and $48$ lab-derived features.

## Create the total first-24-hour lab measurement count

In [15]:
lab_count_source = pd.read_csv(
    DATA_DIR / "lab.csv",
    usecols=[
        "patientunitstayid",
        "labresultoffset"
    ]
)

all_lab_24h = lab_count_source[
    (lab_count_source["patientunitstayid"].isin(final_ids))
    & (lab_count_source["labresultoffset"] >= 0)
    & (lab_count_source["labresultoffset"] <= 1440)
].copy()

lab_measurement_count = (
    all_lab_24h
    .groupby("patientunitstayid")
    .size()
    .reset_index(
        name="total_lab_measurements"
    )
)

lab_measurement_count = (
    final_cohort[
        ["patientunitstayid"]
    ]
    .merge(
        lab_measurement_count,
        on="patientunitstayid",
        how="left"
    )
)

lab_measurement_count[
    "total_lab_measurements"
] = (
    lab_measurement_count[
        "total_lab_measurements"
    ]
    .fillna(0)
    .astype(int)
)

print(
    "Lab measurement count table shape:",
    lab_measurement_count.shape
)

print("\nSummary:")
print(
    lab_measurement_count[
        "total_lab_measurements"
    ].describe()
)

print(
    "\nMost common count:",
    lab_measurement_count[
        "total_lab_measurements"
    ].mode().iloc[0]
)

Lab measurement count table shape: (7880, 2)

Summary:
count    7880.000000
mean       66.255330
std        36.308329
min        19.000000
25%        40.000000
50%        57.000000
75%        82.000000
max       470.000000
Name: total_lab_measurements, dtype: float64

Most common count: 40


#### Observation :
- Counts the total number of laboratory measurements recorded for each patient during the first $24$ hours.
- All $7,880$ patients are included in the count table.
- Patients have a median of $57$ lab measurements with counts ranging from $19$ to $470$.
- The large variation in measurement frequency shows that laboratory monitoring intensity duffers substantially between patients.

## Prepare final-cohort vital-sign trajectories

In [16]:
# Keep vital-sign records only for patients in the final cohort

vital_final = vital_24h[
    vital_24h["patientunitstayid"].isin(final_ids)
].copy()

vital_final = vital_final[
    [
        "patientunitstayid",
        "observationoffset",
        "sao2",
        "heartrate",
        "respiration"
    ]
]

# Fix the patient order for all later FPC features
patient_order = (
    final_cohort["patientunitstayid"]
    .tolist()
)

print(
    "Vital-sign rows in final cohort:",
    len(vital_final)
)

print(
    "Patients represented:",
    vital_final["patientunitstayid"].nunique()
)

print("\nValid measurements:")

print(
    "SpO2:",
    vital_final["sao2"].notna().sum()
)

print(
    "Heart rate:",
    vital_final["heartrate"].notna().sum()
)

print(
    "Respiratory rate:",
    vital_final["respiration"].notna().sum()
)

Vital-sign rows in final cohort: 2075469
Patients represented: 7880

Valid measurements:
SpO2: 1933173
Heart rate: 2071125
Respiratory rate: 2010347


#### Observation :
- Filters the vital-sign data to include only the final $7,880$-patient cohort.
- Keeps the three main vital signs: SpO₂, heart rate, and respiratory rate, along with their measurement times.
- Fixes a consistent patient order, which is important for later functional principal component (FPC) feature creation.
- The final cohort contains over $2.07$ million vital-sign records, with heart rate having the highest number of valid measurements.

## Prepare SpO₂ longitudinal functional data

In [17]:
# Keep observed SpO2 measurements
sao2_data = vital_final[
    vital_final["sao2"].notna()
][
    [
        "patientunitstayid",
        "observationoffset",
        "sao2"
    ]
].copy()

# Sort each patient's measurements by time
sao2_data = sao2_data.sort_values(
    [
        "patientunitstayid",
        "observationoffset"
    ]
)

sao2_groups = sao2_data.groupby(
    "patientunitstayid"
)

sao2_argvals_dict = {}
sao2_values_dict = {}

for i, patient_id in enumerate(patient_order):

    patient_data = sao2_groups.get_group(
        patient_id
    )

    times = patient_data[
        "observationoffset"
    ].to_numpy(dtype=float)

    values = patient_data[
        "sao2"
    ].to_numpy(dtype=float)

    sao2_argvals_dict[i] = DenseArgvals(
        {
            "input_dim_0": times
        }
    )

    sao2_values_dict[i] = values


sao2_fdata = IrregularFunctionalData(
    argvals=IrregularArgvals(
        sao2_argvals_dict
    ),
    values=IrregularValues(
        sao2_values_dict
    )
)

print(
    "Patients in SpO2 functional data:",
    sao2_fdata.n_obs
)

print(
    "Functional dimensions:",
    sao2_fdata.n_dimension
)

print(
    "First patient measurements:",
    len(sao2_values_dict[0])
)

print(
    "First patient time range:",
    sao2_argvals_dict[0]["input_dim_0"].min(),
    "to",
    sao2_argvals_dict[0]["input_dim_0"].max()
)

Patients in SpO2 functional data: 7880
Functional dimensions: 1
First patient measurements: 288
First patient time range: 5.0 to 1440.0


#### Observation :
- Extracts only the available SpO₂ measurements for the final $7,880$ patients.
- Sorts each patient’s SpO₂ readings by time within the first $24$ hours.
- Converts each patient’s measurements into irregular functional data, preserving the actual measurement times instead of forcing equal time intervals.
- All $7,880$ patients are successfully represented, preparing the SpO₂ trajectories for functional PCA (FPCA) in the next step.

## Extract SpO₂ functional principal component features

In [18]:
start = time.time()

# Fit functional PCA to SpO2 trajectories
sao2_fpca = UFPCA(
    n_components=0.90,
    method="covariance"
)

sao2_fpca.fit(
    sao2_fdata,
    method_smoothing="PS"
)

fit_time = time.time() - start


# Calculate patient-level PACE scores
start = time.time()

sao2_scores = sao2_fpca.transform(
    sao2_fdata,
    method="PACE"
)

score_time = time.time() - start


print(
    "SpO2 FPCA fitting time:",
    round(fit_time, 2),
    "seconds"
)

print(
    "SpO2 PACE scoring time:",
    round(score_time, 2),
    "seconds"
)

print(
    "Number of SpO2 components retained:",
    sao2_scores.shape[1]
)

print(
    "SpO2 score matrix shape:",
    sao2_scores.shape
)

print(
    "Missing scores:",
    np.isnan(sao2_scores).sum()
)

SpO2 FPCA fitting time: 558.19 seconds
SpO2 PACE scoring time: 7.97 seconds
Number of SpO2 components retained: 8
SpO2 score matrix shape: (7880, 8)
Missing scores: 0


#### Observation :
- Applies functional PCA (FPCA) to the SpO₂ trajectories to capture the main patterns of oxygen saturation over the first $24$ hours.
- Retains enough components to explain $90\%$ of the variation in SpO₂ patterns.
- Uses the PACE method to generate patient-level FPCA scores from the irregularly measured data.
- $8$ functional principal components are retained for all $7,880$ patients.
- The resulting score matrix has no missing values, making these features ready for downstream modeling or clustering.

## Save the SpO₂ temporal features

In [19]:
# Create readable feature names
sao2_feature_names = [
    f"sao2_fpc{i + 1}"
    for i in range(sao2_scores.shape[1])
]

# Create patient-level feature table
sao2_features = pd.DataFrame(
    sao2_scores,
    columns=sao2_feature_names
)

# Add patient ID
sao2_features.insert(
    0,
    "patientunitstayid",
    patient_order
)

# Save features
sao2_features.to_csv(
    RESULTS_DIR / "sao2_fpc_scores.csv",
    index=False
)

# Save eigenvalues
np.save(
    RESULTS_DIR / "sao2_eigenvalues.npy",
    sao2_fpca.eigenvalues
)

print(
    "SpO2 feature table shape:",
    sao2_features.shape
)

print(
    "Feature names:",
    sao2_feature_names
)

print(
    "Missing values:",
    sao2_features.isna().sum().sum()
)

print(
    "Saved to:",
    RESULTS_DIR / "sao2_fpc_scores.csv"
)

SpO2 feature table shape: (7880, 9)
Feature names: ['sao2_fpc1', 'sao2_fpc2', 'sao2_fpc3', 'sao2_fpc4', 'sao2_fpc5', 'sao2_fpc6', 'sao2_fpc7', 'sao2_fpc8']
Missing values: 0
Saved to: C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\sao2_fpc_scores.csv


#### Observation :
- Converts the $8$ SpO₂ FPCA scores into clearly named patient-level features (`sao2_fpc1` to `sao2_fpc8`).
- Adds the patient ID so each set of FPCA features remains linked to the correct patient.
- Creates a final SpO₂ feature table with $7,880$ patients and $8$ FPCA features.
- Confirms there are no missing values in the generated features.
- Saves both the SpO₂ FPCA scores and eigenvalues for later modeling and interpretation.

## Prepare heart-rate longitudinal functional data

In [20]:
# Keep observed heart-rate measurements
hr_data = vital_final[
    vital_final["heartrate"].notna()
][
    [
        "patientunitstayid",
        "observationoffset",
        "heartrate"
    ]
].copy()

# Sort each patient's measurements by time
hr_data = hr_data.sort_values(
    [
        "patientunitstayid",
        "observationoffset"
    ]
)

hr_groups = hr_data.groupby(
    "patientunitstayid"
)

hr_argvals_dict = {}
hr_values_dict = {}

for i, patient_id in enumerate(patient_order):

    patient_data = hr_groups.get_group(
        patient_id
    )

    times = patient_data[
        "observationoffset"
    ].to_numpy(dtype=float)

    values = patient_data[
        "heartrate"
    ].to_numpy(dtype=float)

    hr_argvals_dict[i] = DenseArgvals(
        {
            "input_dim_0": times
        }
    )

    hr_values_dict[i] = values


hr_fdata = IrregularFunctionalData(
    argvals=IrregularArgvals(
        hr_argvals_dict
    ),
    values=IrregularValues(
        hr_values_dict
    )
)

print(
    "Patients in heart-rate functional data:",
    hr_fdata.n_obs
)

print(
    "Functional dimensions:",
    hr_fdata.n_dimension
)

print(
    "First patient measurements:",
    len(hr_values_dict[0])
)

print(
    "First patient time range:",
    hr_argvals_dict[0]["input_dim_0"].min(),
    "to",
    hr_argvals_dict[0]["input_dim_0"].max()
)

Patients in heart-rate functional data: 7880
Functional dimensions: 1
First patient measurements: 288
First patient time range: 5.0 to 1440.0


#### Observation :
- Extracts the available heart-rate measurements for all $7,880$ patients.
- Sorts each patient’s heart-rate readings by time during the first $24$ hours.
- Converts the measurements into irregular functional data, preserving the actual timing of each observation.
- All $7,880$ patients are successfully represented, preparing their heart-rate trajectories for FPCA analysis.

## Extract heart-rate functional principal component features

In [21]:
start = time.time()

# Fit functional PCA to heart-rate trajectories
hr_fpca = UFPCA(
    n_components=0.90,
    method="covariance"
)

hr_fpca.fit(
    hr_fdata,
    method_smoothing="PS"
)

fit_time = time.time() - start


# Calculate patient-level PACE scores
start = time.time()

hr_scores = hr_fpca.transform(
    hr_fdata,
    method="PACE"
)

score_time = time.time() - start


print(
    "Heart-rate FPCA fitting time:",
    round(fit_time, 2),
    "seconds"
)

print(
    "Heart-rate PACE scoring time:",
    round(score_time, 2),
    "seconds"
)

print(
    "Number of heart-rate components retained:",
    hr_scores.shape[1]
)

print(
    "Heart-rate score matrix shape:",
    hr_scores.shape
)

print(
    "Missing scores:",
    np.isnan(hr_scores).sum()
)

Heart-rate FPCA fitting time: 593.57 seconds
Heart-rate PACE scoring time: 8.02 seconds
Number of heart-rate components retained: 2
Heart-rate score matrix shape: (7880, 2)
Missing scores: 0


#### Observation :
- Applies FPCA to the heart-rate trajectories to capture the main patterns of heart-rate variation during the first $24$ hours.
- Retains enough components to explain $90\%$ of the overall variation.
- Uses the PACE method to calculate patient-level FPCA scores from irregularly recorded heart-rate data.
- Only $2$ functional principal components are needed for all $7,880$ patients.
- The resulting heart-rate score matrix has no missing values, so these features are ready for further analysis.

## Save the heart-rate temporal features

In [22]:
# Create readable feature names
hr_feature_names = [
    f"hr_fpc{i + 1}"
    for i in range(hr_scores.shape[1])
]

# Create patient-level feature table
hr_features = pd.DataFrame(
    hr_scores,
    columns=hr_feature_names
)

# Add patient ID
hr_features.insert(
    0,
    "patientunitstayid",
    patient_order
)

# Save features
hr_features.to_csv(
    RESULTS_DIR / "hr_fpc_scores.csv",
    index=False
)

# Save eigenvalues
np.save(
    RESULTS_DIR / "hr_eigenvalues.npy",
    hr_fpca.eigenvalues
)

print(
    "Heart-rate feature table shape:",
    hr_features.shape
)

print(
    "Feature names:",
    hr_feature_names
)

print(
    "Missing values:",
    hr_features.isna().sum().sum()
)

print(
    "Saved to:",
    RESULTS_DIR / "hr_fpc_scores.csv"
)

Heart-rate feature table shape: (7880, 3)
Feature names: ['hr_fpc1', 'hr_fpc2']
Missing values: 0
Saved to: C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\hr_fpc_scores.csv


#### Observation :
- Converts the $2$ heart-rate FPCA scores into clearly named features: `hr_fpc1` and `hr_fpc2`.
- Adds the patient ID so the FPCA features remain correctly linked to each patient.
- Creates a patient-level heart-rate feature table for all $7,880$ patients.
- Confirms that the generated heart-rate features contain no missing values.
- Saves both the heart-rate FPCA scores and eigenvalues for later analysis and modeling.

## Prepare respiratory-rate longitudinal functional data

In [23]:
# Keep observed respiratory-rate measurements
rr_data = vital_final[
    vital_final["respiration"].notna()
][
    [
        "patientunitstayid",
        "observationoffset",
        "respiration"
    ]
].copy()

# Sort each patient's measurements by time
rr_data = rr_data.sort_values(
    [
        "patientunitstayid",
        "observationoffset"
    ]
)

rr_groups = rr_data.groupby(
    "patientunitstayid"
)

rr_argvals_dict = {}
rr_values_dict = {}

for i, patient_id in enumerate(patient_order):

    patient_data = rr_groups.get_group(
        patient_id
    )

    times = patient_data[
        "observationoffset"
    ].to_numpy(dtype=float)

    values = patient_data[
        "respiration"
    ].to_numpy(dtype=float)

    rr_argvals_dict[i] = DenseArgvals(
        {
            "input_dim_0": times
        }
    )

    rr_values_dict[i] = values


rr_fdata = IrregularFunctionalData(
    argvals=IrregularArgvals(
        rr_argvals_dict
    ),
    values=IrregularValues(
        rr_values_dict
    )
)

print(
    "Patients in respiratory-rate functional data:",
    rr_fdata.n_obs
)

print(
    "Functional dimensions:",
    rr_fdata.n_dimension
)

print(
    "First patient measurements:",
    len(rr_values_dict[0])
)

print(
    "First patient time range:",
    rr_argvals_dict[0]["input_dim_0"].min(),
    "to",
    rr_argvals_dict[0]["input_dim_0"].max()
)

Patients in respiratory-rate functional data: 7880
Functional dimensions: 1
First patient measurements: 75
First patient time range: 1070.0 to 1440.0


#### Observation :
- Extracts the available respiratory-rate measurements for all $7,880$ patients.
- Sorts each patient’s readings by time within the first $24$ hours.
- Converts the measurements into irregular functional data, keeping the actual timing of each respiratory-rate observation.
- All $7,880$ patients are successfully represented, preparing the respiratory-rate trajectories for FPCA analysis.

## Extract respiratory-rate functional principal component features

In [24]:
start = time.time()

# Fit functional PCA to respiratory-rate trajectories
rr_fpca = UFPCA(
    n_components=0.90,
    method="covariance"
)

rr_fpca.fit(
    rr_fdata,
    method_smoothing="PS"
)

fit_time = time.time() - start


# Calculate patient-level PACE scores
start = time.time()

rr_scores = rr_fpca.transform(
    rr_fdata,
    method="PACE"
)

score_time = time.time() - start


print(
    "Respiratory-rate FPCA fitting time:",
    round(fit_time, 2),
    "seconds"
)

print(
    "Respiratory-rate PACE scoring time:",
    round(score_time, 2),
    "seconds"
)

print(
    "Number of respiratory-rate components retained:",
    rr_scores.shape[1]
)

print(
    "Respiratory-rate score matrix shape:",
    rr_scores.shape
)

print(
    "Missing scores:",
    np.isnan(rr_scores).sum()
)

Respiratory-rate FPCA fitting time: 568.19 seconds
Respiratory-rate PACE scoring time: 7.99 seconds
Number of respiratory-rate components retained: 4
Respiratory-rate score matrix shape: (7880, 4)
Missing scores: 0


#### Observation :
- Applies FPCA to respiratory-rate trajectories from the first $24$ hours.
- Retains enough components to explain $90\%$ of the variation in respiratory-rate patterns.
- Uses the PACE method to generate patient-level FPCA scores from irregularly recorded measurements.
- $4$ functional principal components are retained for all $7,880$ patients.
- The resulting score matrix has no missing values, so these features are ready for downstream analysis.

## Save the respiratory-rate temporal features

In [25]:
# Create readable feature names
rr_feature_names = [
    f"rr_fpc{i + 1}"
    for i in range(rr_scores.shape[1])
]

# Create patient-level feature table
rr_features = pd.DataFrame(
    rr_scores,
    columns=rr_feature_names
)

# Add patient ID
rr_features.insert(
    0,
    "patientunitstayid",
    patient_order
)

# Save features
rr_features.to_csv(
    RESULTS_DIR / "rr_fpc_scores.csv",
    index=False
)

# Save eigenvalues
np.save(
    RESULTS_DIR / "rr_eigenvalues.npy",
    rr_fpca.eigenvalues
)

print(
    "Respiratory-rate feature table shape:",
    rr_features.shape
)

print(
    "Feature names:",
    rr_feature_names
)

print(
    "Missing values:",
    rr_features.isna().sum().sum()
)

print(
    "Saved to:",
    RESULTS_DIR / "rr_fpc_scores.csv"
)

Respiratory-rate feature table shape: (7880, 5)
Feature names: ['rr_fpc1', 'rr_fpc2', 'rr_fpc3', 'rr_fpc4']
Missing values: 0
Saved to: C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\rr_fpc_scores.csv


#### Observation :
- Converts the $4$ respiratory-rate FPCA scores into clearly named features: `rr_fpc1` to `rr_fpc4`.
- Adds the patient ID so each respiratory-rate pattern remains linked to the correct patient.
- Creates a feature table for all $7,880$ patients with no missing values.
- Saves the respiratory-rate FPCA scores and eigenvalues for later analysis and clustering.

## Combine all vital-sign temporal features

In [26]:
vital_fpc_features = (
    sao2_features
    .merge(
        hr_features,
        on="patientunitstayid",
        how="inner"
    )
    .merge(
        rr_features,
        on="patientunitstayid",
        how="inner"
    )
)

print(
    "Combined vital feature table shape:",
    vital_fpc_features.shape
)

print(
    "Patients represented:",
    vital_fpc_features["patientunitstayid"].nunique()
)

print(
    "Number of vital-sign features:",
    vital_fpc_features.shape[1] - 1
)

print(
    "Missing values:",
    vital_fpc_features.isna().sum().sum()
)

print("\nFeature names:")
print(
    vital_fpc_features
    .columns
    .drop("patientunitstayid")
    .tolist()
)

Combined vital feature table shape: (7880, 15)
Patients represented: 7880
Number of vital-sign features: 14
Missing values: 0

Feature names:
['sao2_fpc1', 'sao2_fpc2', 'sao2_fpc3', 'sao2_fpc4', 'sao2_fpc5', 'sao2_fpc6', 'sao2_fpc7', 'sao2_fpc8', 'hr_fpc1', 'hr_fpc2', 'rr_fpc1', 'rr_fpc2', 'rr_fpc3', 'rr_fpc4']


#### Observation :
- Combines the SpO₂, heart-rate, and respiratory-rate FPCA features into one patient-level vital-sign feature table.
- The combined dataset contains all $7,880$ patients and $14$ vital-sign features.
- No patients are lost during the merge, confirming that all three FPCA feature sets cover the same final cohort.
- There are no missing values, so the combined vital-sign features are complete and ready to be merged with the other patient features.

## Combine all feature blocks

In [27]:
feature_data = (
    demo_features
    .merge(
        lab_summary,
        on="patientunitstayid",
        how="inner"
    )
    .merge(
        lab_measurement_count,
        on="patientunitstayid",
        how="inner"
    )
    .merge(
        vital_fpc_features,
        on="patientunitstayid",
        how="inner"
    )
)

print(
    "Combined feature table shape:",
    feature_data.shape
)

print(
    "Patients represented:",
    feature_data["patientunitstayid"].nunique()
)

print(
    "Duplicate patient IDs:",
    feature_data["patientunitstayid"]
    .duplicated()
    .sum()
)

print(
    "\nTotal missing values:",
    feature_data.isna().sum().sum()
)

print("\nColumns with missing values:")

missing_columns = (
    feature_data
    .isna()
    .sum()
)

print(
    missing_columns[
        missing_columns > 0
    ]
)

Combined feature table shape: (7880, 70)
Patients represented: 7880
Duplicate patient IDs: 0

Total missing values: 258

Columns with missing values:
admissionheight     67
admissionweight    191
dtype: int64


#### Observation :
- Combines the demographic, laboratory summary, laboratory measurement count, and vital-sign FPCA features into one final patient-level dataset.
- The combined table contains $7,880$ unique patients and $70$ columns, with no duplicate patient IDs.
- Most features are complete; missing values remain only in admission height ($67$) and admission weight ($191$).
- This block creates the complete feature set that will be cleaned and prepared for the final analysis or clustering.

## Create the complete-case feature dataset

In [28]:
feature_data_complete = (
    feature_data
    .dropna()
    .copy()
)

print(
    "Patients before removing incomplete records:",
    len(feature_data)
)

print(
    "Patients removed:",
    len(feature_data) - len(feature_data_complete)
)

print(
    "Patients remaining:",
    len(feature_data_complete)
)

print(
    "Unique patients:",
    feature_data_complete[
        "patientunitstayid"
    ].nunique()
)

print(
    "Remaining missing values:",
    feature_data_complete
    .isna()
    .sum()
    .sum()
)

print(
    "Final complete-case table shape:",
    feature_data_complete.shape
)

Patients before removing incomplete records: 7880
Patients removed: 232
Patients remaining: 7648
Unique patients: 7648
Remaining missing values: 0
Final complete-case table shape: (7648, 70)


#### Observation :
- Removes patients with any remaining missing values from the combined feature dataset.
- The missing values are mainly from admission height and admission weight, so only complete patient records are retained.
- This reduces the dataset from $7,880$ to $7,626$ patients.
- The resulting dataset has no missing values, making it ready for encoding and scaling.

## Encode categorical variables

We now convert gender, ethnicity and ICU type into numeric variables so every clustering feature is numerical

In [29]:
feature_data_encoded = feature_data_complete.copy()

# Encode gender as binary
feature_data_encoded["gender"] = (
    feature_data_encoded["gender"]
    .map({
        "Male": 0,
        "Female": 1
    })
)

# One-hot encode ethnicity and ICU type
feature_data_encoded = pd.get_dummies(
    feature_data_encoded,
    columns=[
        "ethnicity",
        "unittype"
    ],
    drop_first=False,
    dtype=int
)

print(
    "Encoded dataset shape:",
    feature_data_encoded.shape
)

print(
    "Number of patient records:",
    len(feature_data_encoded)
)

print(
    "Number of clustering features:",
    feature_data_encoded.shape[1] - 1
)

print(
    "Missing values:",
    feature_data_encoded.isna().sum().sum()
)

print(
    "\nNon-numeric feature columns:"
)

print(
    feature_data_encoded
    .drop(columns="patientunitstayid")
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

Encoded dataset shape: (7648, 81)
Number of patient records: 7648
Number of clustering features: 80
Missing values: 0

Non-numeric feature columns:
[]


#### Observation :
- Converts all categorical variables into numeric form so the dataset can be used for clustering.
- Encodes gender as binary and uses one-hot encoding for ethnicity and ICU type.
- The encoded dataset contains $7,648$ patients and $80$ clustering features.
- Confirms there are no missing values or non-numeric feature columns remaining.
- This leaves the dataset fully prepared for the next preprocessing step, such as feature scaling.

## Min-max scale the clustering features

In [30]:
# Separate patient ID from clustering features
patient_ids = feature_data_encoded[
    "patientunitstayid"
].copy()

X = feature_data_encoded.drop(
    columns="patientunitstayid"
).copy()


# Min-max scaling
scaler = MinMaxScaler()

X_scaled_array = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled_array,
    columns=X.columns,
    index=X.index
)


# Add patient ID back
scaled_feature_data = X_scaled.copy()

scaled_feature_data.insert(
    0,
    "patientunitstayid",
    patient_ids.values
)


print(
    "Scaled dataset shape:",
    scaled_feature_data.shape
)

print(
    "Number of clustering features:",
    X_scaled.shape[1]
)

print(
    "Minimum feature value:",
    X_scaled.min().min()
)

print(
    "Maximum feature value:",
    X_scaled.max().max()
)

print(
    "Missing values:",
    X_scaled.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(X_scaled.to_numpy()).sum()
)

print(
    "Features with zero variance:",
    (X.nunique() <= 1).sum()
)

Scaled dataset shape: (7648, 81)
Number of clustering features: 80
Minimum feature value: 0.0
Maximum feature value: 1.0000000000000002
Missing values: 0
Infinite values: 0
Features with zero variance: 0


#### Observation :
- Separates the patient ID from the $80$ clustering features so the identifier is not included in scaling.
- Applies Min-Max scaling, transforming all clustering features to approximately the $0–1$ range.
- Adds the patient ID back after scaling, producing a final dataset with $7,648$ patients and $80$ scaled features.
- Confirms there are no missing or infinite values after scaling.
- No feature has zero variance, meaning every feature contributes some variation across patients.
- This produces a clean, standardized dataset ready for clustering analysis.

## Save the final datasets

In [31]:
# Save complete-case features before encoding
feature_data_complete.to_csv(
    RESULTS_DIR / "feature_data_complete.csv",
    index=False
)

# Save encoded but unscaled features
feature_data_encoded.to_csv(
    RESULTS_DIR / "feature_data_encoded.csv",
    index=False
)

# Save final scaled clustering dataset
scaled_feature_data.to_csv(
    RESULTS_DIR / "feature_data_scaled.csv",
    index=False
)

# Save clustering feature names in exact column order
feature_names = X_scaled.columns.tolist()

pd.Series(
    feature_names,
    name="feature"
).to_csv(
    RESULTS_DIR / "clustering_feature_names.csv",
    index=False
)

# Save combined vital-sign features
vital_fpc_features.to_csv(
    RESULTS_DIR / "vital_fpc_features.csv",
    index=False
)

print("Saved files:")

print(
    RESULTS_DIR / "feature_data_complete.csv"
)

print(
    RESULTS_DIR / "feature_data_encoded.csv"
)

print(
    RESULTS_DIR / "feature_data_scaled.csv"
)

print(
    RESULTS_DIR / "clustering_feature_names.csv"
)

print(
    RESULTS_DIR / "vital_fpc_features.csv"
)

print(
    "\nFinal patients:",
    len(scaled_feature_data)
)

print(
    "Final clustering features:",
    len(feature_names)
)

Saved files:
C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\feature_data_complete.csv
C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\feature_data_encoded.csv
C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\feature_data_scaled.csv
C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\clustering_feature_names.csv
C:\Users\samsa\Documents\ICU Clustering\results\module_2\dataset_creation\vital_fpc_features.csv

Final patients: 7648
Final clustering features: 80


#### Observation :
- Saves the main datasets at different preprocessing stages: complete-case, encoded, and scaled features.
- Saves the exact $80$ clustering feature names and their order, which is important for reproducibility in later modeling.
- Also saves the combined vital-sign FPCA features separately for reference or additional analysis.
- Confirms that the final clustering dataset contains $7,648$ patients and $80$ features.
- This block ensures all processed outputs are stored consistently for the next stages of the clustering workflow.

## Final validation of the saved dataset

In [32]:
# Reload the final saved dataset
final_saved_data = pd.read_csv(
    RESULTS_DIR / "feature_data_scaled.csv"
)

saved_feature_names = pd.read_csv(
    RESULTS_DIR / "clustering_feature_names.csv"
)["feature"].tolist()

# Separate ID and features
X_check = final_saved_data.drop(
    columns="patientunitstayid"
)

print(
    "Reloaded dataset shape:",
    final_saved_data.shape
)

print(
    "Unique patients:",
    final_saved_data["patientunitstayid"].nunique()
)

print(
    "Duplicate patient IDs:",
    final_saved_data["patientunitstayid"]
    .duplicated()
    .sum()
)

print(
    "Number of clustering features:",
    X_check.shape[1]
)

print(
    "Feature order matches saved feature list:",
    saved_feature_names == X_check.columns.tolist()
)

print(
    "Missing values:",
    X_check.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(X_check.to_numpy()).sum()
)

print(
    "Minimum feature value:",
    X_check.min().min()
)

print(
    "Maximum feature value:",
    X_check.max().max()
)

Reloaded dataset shape: (7648, 81)
Unique patients: 7648
Duplicate patient IDs: 0
Number of clustering features: 80
Feature order matches saved feature list: True
Missing values: 0
Infinite values: 0
Minimum feature value: 0.0
Maximum feature value: 1.0000000000000002


#### Observation :
- Reloads the saved scaled dataset and feature-name list to verify that the files were stored correctly.
- Confirms the dataset has the expected number of patients and clustering features, with no duplicate patient IDs.
- Checks that the feature order exactly matches the saved feature list, which is important for reproducible clustering.
- Verifies that there are no missing or infinite values in the final feature matrix.
- Confirms that the scaled feature values remain within the expected $0–1$ range.
- This serves as the final quality-control check before using the dataset for clustering or further analysis.